# Code for the training of the $C_4$-equivariant NN - 6.4  Architecture for **$\pi/2$ rotations**:

## Applies dropout once only and uses Historical statistics for OrbitBN 

- 4 layers
    ```python
  Input (250^2)
    → W1 (100×250^2)      # G-equivariant: → W1 (100 x 250^2)      # W_1 G = phi_1 W_1 and phi_1 is the 100 x 100 permutation, G loaded from MTX_FILENAME = "../Files/BetterACpermutation_matrix_250_90.000.mtx" and G_10 =  MTX_FILENAME = "../Files/BetterACpermutation_matrix_10_90.000.mtx" (W1 is 100 x 250^2) 
    → OrbitBatchNorm (G)
    → GELU
    → OrbitDropout (G)
    → W2 (8×100)         # G → Phi intertwiner:  W_2 G_10 = phi_2 W_2, phi_2 is the a 8 x 8 permutation (W2 is 8 x 100)
    → OrbitBatchNorm (Φ)
    → GELU                 # ← NEW activation
    → W3 (8×8)            # ← NEW Φ-equivariant layer
    → OrbitBatchNorm (Φ)   # ← NEW batch norm
    → Sigma2 (L + norm-based)
    → Output (2)
  ```
- Orbit-based:
   - Batch-norm
   - Drop-out
- Other Regularisation:
   - Weight decay : X
   - L1/L2 on weight: X

### Info saved:

```python

experiments/exp_YYYYMMDD_HHMMSS/
├── config.json              # Full config including regularization details
├── training_log.csv         # Loss per epoch
├── learning_curves.png      # Train/val loss plot
├── model_best.pt            # Best checkpoint (includes BN states)
├── model_final.pt           # Final checkpoint
├── val_metrics.json         # Validation metrics (per-output)
├── val_predictions.csv      # Validation predictions
├── val_pred_vs_true.png     # Validation scatter plot
├── val_residuals.png        # Validation residuals
├── test_metrics.json        # Test metrics
├── test_predictions.csv     # Test predictions
├── test_pred_vs_true.png    # Test scatter plot
├── test_residuals.png       # Test residuals
├── equivariance_check.json  # Equivariance verification
└── notes.md                 # Pre-filled notes template

```

In [ ]:
#!/usr/bin/env python3
# equivariant_experiment_final_architecture_modified_full_save.py
# Structure:
# Input(250^2) -> [W1 -> BN(G_hid) -> GELU -> Drop] -> [W2 -> BN(Phi) -> GELU] -> [W3 -> BN(Phi)] -> Sigma2 -> y

import os
import glob
import re
import math
import json
import datetime
import csv
from typing import List, Tuple, Sequence
import numpy as np
from scipy.io import mmread
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from torchvision.io import read_image
from tqdm import tqdm
import matplotlib.pyplot as plt

# ----------------------------
# Global config
# ----------------------------
torch.set_default_dtype(torch.float32)
SEED = 317 #316 #315 #314
np.random.seed(SEED)
torch.manual_seed(SEED)

# Architecture specific files
MTX_INPUT_FILENAME = "../Files/BetterACpermutation_matrix_250_90.000.mtx" 
MTX_HIDDEN_FILENAME = "../Files/BetterACpermutation_matrix_10_90.000.mtx"

## Modify:
IMAGE_DIR = "output_ellipses/SquareNew_Train250x250Images100Ellipses" 
TEST_DIR = "output_ellipses/MoreSquareTestData"

MODEL_NAME = "6.4_Model_HighDim_W1_64x62500_W2_8x100"

N_EPOCHS = 25
LR = 1e-3
BATCH_SIZE = 32
TRAIN_SPLIT = 0.8
DROPOUT_RATE = 0.1
TRAINABLE_PARAMS_RATIO = 1.0 

NUM_WORKERS = 0 if not torch.cuda.is_available() else 4
PIN_MEMORY = torch.cuda.is_available()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ----------------------------
# Experiment Tracking Setup
# ----------------------------
def create_experiment_folder(base_dir="experiments"):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    exp_name = f"({MODEL_NAME})exp_{timestamp}"
    exp_dir = os.path.join(base_dir, exp_name)
    os.makedirs(exp_dir, exist_ok=True)
    return exp_dir

def save_config(exp_dir: str, config: dict):
    config_path = os.path.join(exp_dir, "config.json")
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)
    print(f"Saved config to {config_path}")

def save_training_log(exp_dir: str, log: list):
    if len(log) == 0: return
    log_path = os.path.join(exp_dir, "training_log.csv")
    with open(log_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=log[0].keys())
        writer.writeheader()
        writer.writerows(log)
    print(f"Saved training log to {log_path}")

def save_predictions(exp_dir: str, predictions: dict, filename: str):
    pred_path = os.path.join(exp_dir, filename)
    with open(pred_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['true_q11', 'true_q12', 'pred_q11', 'pred_q12', 'error_q11', 'error_q12'])
        for i in range(len(predictions['true'])):
            true = predictions['true'][i]
            pred = predictions['pred'][i]
            writer.writerow([
                true[0], true[1], 
                pred[0], pred[1],
                abs(true[0] - pred[0]), abs(true[1] - pred[1])
            ])
    print(f"Saved predictions to {pred_path}")

def plot_learning_curves(exp_dir: str, log: list):
    if len(log) == 0: return
    epochs = [entry['epoch'] for entry in log]
    train_losses = [entry['train_loss'] for entry in log]
    val_losses = [entry['val_loss'] for entry in log]
    
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_losses, 'b-', label='Train Loss', marker='o')
    plt.plot(epochs, val_losses, 'r-', label='Val Loss', marker='s')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title('Learning Curves')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(exp_dir, "learning_curves.png"), dpi=150)
    plt.close()

def plot_predictions(exp_dir: str, predictions: dict, prefix: str = ""):
    true = np.array(predictions['true'])
    pred = np.array(predictions['pred'])
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for i, label in enumerate(['Q11', 'Q12']):
        ax = axes[i]
        ax.scatter(true[:, i], pred[:, i], alpha=0.5, s=10)
        min_val = min(true[:, i].min(), pred[:, i].min())
        max_val = max(true[:, i].max(), pred[:, i].max())
        ax.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect')
        ax.set_xlabel(f'True {label}')
        ax.set_ylabel(f'Predicted {label}')
        ax.set_title(f'{prefix}{label}: Predicted vs True')
        ax.legend()
        ax.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(exp_dir, f"{prefix}pred_vs_true.png"), dpi=150)
    plt.close()

def plot_residuals(exp_dir: str, predictions: dict, prefix: str = ""):
    true = np.array(predictions['true'])
    pred = np.array(predictions['pred'])
    residuals = pred - true
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for i, label in enumerate(['Q11', 'Q12']):
        ax = axes[i]
        ax.hist(residuals[:, i], bins=50, edgecolor='black', alpha=0.7)
        ax.axvline(x=0, color='r', linestyle='--')
        ax.set_xlabel(f'Residual (Pred - True)')
        ax.set_ylabel('Frequency')
        ax.set_title(f'{prefix}{label} Residuals')
        ax.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(exp_dir, f"{prefix}residuals.png"), dpi=150)
    plt.close()

def compute_metrics(predictions: dict) -> dict:
    true = np.array(predictions['true'])
    pred = np.array(predictions['pred'])
    metrics = {}
    for i, label in enumerate(['q11', 'q12']):
        t, p = true[:, i], pred[:, i]
        mse = np.mean((t - p) ** 2)
        metrics[f'{label}_mse'] = float(mse)
        metrics[f'{label}_mae'] = float(np.mean(np.abs(t - p)))
        metrics[f'{label}_max_error'] = float(np.max(np.abs(t - p)))
    combined_mse = np.mean((true - pred) ** 2)
    metrics['combined_mse'] = float(combined_mse)
    return metrics

def save_metrics(exp_dir: str, metrics: dict, filename: str):
    metrics_path = os.path.join(exp_dir, filename)
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f"Saved metrics to {metrics_path}")

def save_notes(exp_dir: str, notes: str):
    notes_path = os.path.join(exp_dir, "notes.md")
    with open(notes_path, 'w') as f:
        f.write(notes)

# ----------------------------
# Utils: Permutations & Cycles
# ----------------------------
def load_permutation_sigma(filename: str, fallback_N: int = 100) -> Tuple[List[int], int]:
    try:
        G_sparse = mmread(filename).tocsc()
        N = G_sparse.shape[0]
        sigma = [int(G_sparse[:, j].indices[0]) for j in range(N)]
        print(f"Loaded permutation from {filename} (N={N})")
        return sigma, N
    except Exception as e:
        print(f"Warning: Could not load {filename} ({e}). Generating mock permutation for N={fallback_N}.")
        N = fallback_N
        sigma = list(range(N))
        if N > 1: sigma = sigma[1:] + sigma[:1]
        return sigma, N
    
def cycles_from_sigma(sigma: Sequence[int]) -> List[np.ndarray]:
    N = len(sigma)
    seen = np.zeros(N, dtype=bool)
    cycles = []
    for i in range(N):
        if seen[i]: continue
        cur = i
        cyc = []
        while not seen[cur]:
            cyc.append(cur)
            seen[cur] = True
            cur = sigma[cur]
        cycles.append(np.array(cyc, dtype=np.int64))
    return cycles

def build_orbit_descriptors_from_cycles(left_cycles, right_cycles):
    descriptors = []
    for ia, A in enumerate(left_cycles):
        La = len(A)
        for ib, B in enumerate(right_cycles):
            Lb = len(B)
            g = math.gcd(La, Lb)
            orbit_len = (La * Lb) // g
            for r in range(g):
                descriptors.append((ia, ib, r, orbit_len, La, Lb))
    return descriptors

def expand_descriptor_to_indices(desc, left_cycles, right_cycles):
    ia, ib, r, orbit_len, La, Lb = desc
    A = left_cycles[ia]
    B = right_cycles[ib]
    t = np.arange(orbit_len)
    rows = A[(r + t) % La]
    cols = B[t % Lb]
    return rows, cols

# ----------------------------
# Equivariant Layers
# ----------------------------
class OrbitBatchNorm(nn.Module):
    def __init__(self, cycles: List[np.ndarray], num_features: int, eps=1e-5, momentum=0.1):
        super().__init__()
        self.eps = eps
        self.momentum = momentum
        self.num_cycles = len(cycles)
        
        cycle_indices = torch.zeros(num_features, dtype=torch.long)
        for i, cyc in enumerate(cycles):
            cycle_indices[cyc] = i
        self.register_buffer('cycle_indices', cycle_indices)
        
        cycle_counts = torch.zeros(self.num_cycles, dtype=torch.float32)
        for i, cyc in enumerate(cycles):
            cycle_counts[i] = len(cyc)
        self.register_buffer('cycle_counts', cycle_counts)
        
        self.gamma = nn.Parameter(torch.ones(self.num_cycles))
        self.beta = nn.Parameter(torch.zeros(self.num_cycles))
        
        self.register_buffer('running_mean', torch.zeros(self.num_cycles))
        self.register_buffer('running_var', torch.ones(self.num_cycles))

    def forward(self, x):
        B, N = x.shape
        if self.training:
            batch_sum = x.sum(dim=0)
            batch_sq_sum = (x ** 2).sum(dim=0)
            
            orbit_sum = torch.zeros(self.num_cycles, device=x.device)
            orbit_sq_sum = torch.zeros(self.num_cycles, device=x.device)
            
            orbit_sum.scatter_add_(0, self.cycle_indices, batch_sum)
            orbit_sq_sum.scatter_add_(0, self.cycle_indices, batch_sq_sum)
            
            orbit_total_count = self.cycle_counts * B
            batch_mean = orbit_sum / orbit_total_count
            batch_var = (orbit_sq_sum / orbit_total_count) - (batch_mean ** 2)
            
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * batch_mean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * batch_var
            
            mean, std = batch_mean, torch.sqrt(batch_var + self.eps)
        else:
            mean, std = self.running_mean, torch.sqrt(self.running_var + self.eps)
        
        mean_expanded = mean[self.cycle_indices]
        std_expanded = std[self.cycle_indices]
        gamma_expanded = self.gamma[self.cycle_indices]
        beta_expanded = self.beta[self.cycle_indices]
        
        x_norm = (x - mean_expanded.unsqueeze(0)) / std_expanded.unsqueeze(0)
        return x_norm * gamma_expanded.unsqueeze(0) + beta_expanded.unsqueeze(0)

class OrbitDropout(nn.Module):
    def __init__(self, p: float, cycles: List[np.ndarray], num_features: int):
        super().__init__()
        self.p = p
        self.num_cycles = len(cycles)
        cycle_indices = torch.zeros(num_features, dtype=torch.long)
        for i, cyc in enumerate(cycles):
            cycle_indices[cyc] = i
        self.register_buffer('cycle_indices', cycle_indices)

    def forward(self, x):
        if not self.training or self.p == 0:
            return x
        B = x.shape[0]
        mask_params = torch.full((B, self.num_cycles), 1 - self.p, device=x.device)
        mask_coarse = torch.bernoulli(mask_params)
        mask = mask_coarse[:, self.cycle_indices]
        return x * mask / (1 - self.p)

def make_batched_matvec(rows_flat, cols_flat, pidx_flat, params):
    rows = rows_flat
    cols = cols_flat
    pidx = pidx_flat
    out_dim = int(rows.max().item()) + 1
    
    def matvec(X):
        X_cols = torch.index_select(X, 1, cols)
        vals = params[pidx].unsqueeze(0)
        weighted = X_cols * vals
        Y = torch.zeros(X.size(0), out_dim, device=X.device, dtype=X.dtype)
        Y.scatter_add_(1, rows.unsqueeze(0).expand(X.size(0), -1), weighted)
        return Y
    return matvec

class Sigma2(nn.Module):
    def __init__(self, L_matrix: torch.Tensor):
        super().__init__()
        self.register_buffer('L', L_matrix)

    def forward(self, y):
        z = y @ self.L.t()
        norm = torch.norm(z, p=2, dim=1, keepdim=True)
        safe_norm = norm + 1e-8
        scale = 0.5 * torch.tanh(norm) / safe_norm
        return z * scale

# ----------------------------
# Dataset
# ----------------------------
class EllipseDataset(Dataset):
    def __init__(self, img_dir: str, preload: bool = True):
        super().__init__()
        self.img_dir = img_dir
        self.img_files = sorted(glob.glob(os.path.join(img_dir, '*.png')))
        self.labels = []
        valid_files = []
        for f in self.img_files:
            filename = os.path.basename(f)
            match = re.search(r"Q11_([\d\.\-]+)_Q12_([\d\.\-]+)\.png", filename)
            if match:
                q11 = float(match.group(1))
                q12 = float(match.group(2))
                self.labels.append(torch.tensor([q11, q12], dtype=torch.float32))
                valid_files.append(f)
        self.img_files = valid_files
        self._preloaded = []
        if preload and len(self.img_files) > 0:
            for f in tqdm(self.img_files, desc=f"Preloading {os.path.basename(img_dir)}"):
                img = read_image(f).float() / 255.0
                img = transforms.functional.normalize(img, mean=[0.5], std=[0.5])
                self._preloaded.append(img.flatten())

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        if self._preloaded:
            return self._preloaded[idx], self.labels[idx]
        img = read_image(self.img_files[idx]).float() / 255.0
        img = transforms.functional.normalize(img, mean=[0.5], std=[0.5])
        return img.flatten(), self.labels[idx]
    
    def get_label_statistics(self) -> dict:
        if len(self.labels) == 0: return {}
        labels = torch.stack(self.labels).numpy()
        return {
            'q11_mean': float(labels[:, 0].mean()), 'q11_std': float(labels[:, 0].std()),
            'q12_mean': float(labels[:, 1].mean()), 'q12_std': float(labels[:, 1].std()),
            'count': len(labels)
        }

# ----------------------------
# Model 
# ----------------------------
class EquivariantModel(nn.Module):
    def __init__(self, 
                 rows_w1, cols_w1, pidx_w1, bn1, drop1,
                 rows_w2, cols_w2, pidx_w2, bn2, 
                 rows_w3, cols_w3, pidx_w3, bn3,
                 sigma2_layer):
        super().__init__()
        
        # Layer 1
        self.register_buffer('rows_w1', rows_w1)
        self.register_buffer('cols_w1', cols_w1)
        self.register_buffer('pidx_w1', pidx_w1)
        self.bn1 = bn1
        self.drop1 = drop1
        
        # Layer 2
        self.register_buffer('rows_w2', rows_w2)
        self.register_buffer('cols_w2', cols_w2)
        self.register_buffer('pidx_w2', pidx_w2)
        self.bn2 = bn2
        
        # Layer 3
        self.register_buffer('rows_w3', rows_w3)
        self.register_buffer('cols_w3', cols_w3)
        self.register_buffer('pidx_w3', pidx_w3)
        self.bn3 = bn3
        
        self.sigma2 = sigma2_layer
        
    def forward(self, x, params_w1, params_w2, params_w3):
        # L1
        matvec_1 = make_batched_matvec(self.rows_w1, self.cols_w1, self.pidx_w1, params_w1)
        h = matvec_1(x)
        h = self.bn1(h)
        h = F.gelu(h)  
        h = self.drop1(h) 
        
        # L2
        matvec_2 = make_batched_matvec(self.rows_w2, self.cols_w2, self.pidx_w2, params_w2)
        z = matvec_2(h)
        z = self.bn2(z)
        z = F.gelu(z) 
        
        # L3
        matvec_3 = make_batched_matvec(self.rows_w3, self.cols_w3, self.pidx_w3, params_w3)
        u = matvec_3(z)
        u = self.bn3(u)
        
        # Out
        out = self.sigma2(u)
        return out

# ----------------------------
# Evaluation Function
# ----------------------------
def evaluate_model(model, dataloader, w1_params, w2_params, w3_params, device):
    model.eval()
    all_true = []
    all_pred = []
    total_loss = 0.0
    loss_fn = nn.MSELoss()
    
    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            x = x_batch.to(device)
            y = y_batch.to(device)
            pred = model(x, w1_params, w2_params, w3_params)
            total_loss += loss_fn(pred, y).item()
            all_true.extend(y.cpu().numpy().tolist())
            all_pred.extend(pred.cpu().numpy().tolist())
    
    avg_loss = total_loss / len(dataloader) if len(dataloader) > 0 else 0
    return {'true': all_true, 'pred': all_pred, 'avg_loss': avg_loss}

# ----------------------------
# Main
# ----------------------------
def main():
    exp_dir = create_experiment_folder()
    print(f"Experiment folder: {exp_dir}")
    print(f"Device: {DEVICE}")
    print("Structure: Input(250^2) -> [W1 -> BN -> GELU -> Drop] -> [W2 -> BN -> GELU] -> [W3 -> BN] -> Sigma2")
    
    # 1. Groups & Cycles
    sigma_in, N_in = load_permutation_sigma(MTX_INPUT_FILENAME, fallback_N=250*250)
    cycles_G_in = cycles_from_sigma(sigma_in)
    
    sigma_hid, N_hid = load_permutation_sigma(MTX_HIDDEN_FILENAME, fallback_N=100)
    cycles_G_hid = cycles_from_sigma(sigma_hid)
    
    cycles_Phi = [
        np.array([0, 1, 2, 3], dtype=np.int64),
        np.array([4, 5, 6, 7], dtype=np.int64)
    ]
    
    # 2. Build W1 (250^2 -> 100)
    print("Building W1 descriptors...")
    desc_w1 = build_orbit_descriptors_from_cycles(cycles_G_hid, cycles_G_in)
    r1, c1, pi1 = [], [], []
    for i, desc in enumerate(desc_w1):
        r, c = expand_descriptor_to_indices(desc, cycles_G_hid, cycles_G_in)
        r1.append(r); c1.append(c); pi1.append(np.full(len(r), i, dtype=np.int64))
    
    rows_w1 = torch.tensor(np.concatenate(r1), dtype=torch.long, device=DEVICE)
    cols_w1 = torch.tensor(np.concatenate(c1), dtype=torch.long, device=DEVICE)
    pidx_w1 = torch.tensor(np.concatenate(pi1), dtype=torch.long, device=DEVICE)
    
    w1_params = nn.Parameter(torch.randn(len(desc_w1), device=DEVICE) * 0.01)
    bn1 = OrbitBatchNorm(cycles_G_hid, N_hid).to(DEVICE)
    drop1 = OrbitDropout(DROPOUT_RATE, cycles_G_hid, N_hid).to(DEVICE)
    
    # 3. Build W2 (100 -> 8)
    print("Building W2 descriptors...")
    desc_w2 = build_orbit_descriptors_from_cycles(cycles_Phi, cycles_G_hid)
    r2, c2, pi2 = [], [], []
    for i, desc in enumerate(desc_w2):
        r, c = expand_descriptor_to_indices(desc, cycles_Phi, cycles_G_hid)
        r2.append(r); c2.append(c); pi2.append(np.full(len(r), i, dtype=np.int64))
        
    rows_w2 = torch.tensor(np.concatenate(r2), dtype=torch.long, device=DEVICE)
    cols_w2 = torch.tensor(np.concatenate(c2), dtype=torch.long, device=DEVICE)
    pidx_w2 = torch.tensor(np.concatenate(pi2), dtype=torch.long, device=DEVICE)
    
    w2_params = nn.Parameter(torch.randn(len(desc_w2), device=DEVICE) * 0.01)
    bn2 = OrbitBatchNorm(cycles_Phi, 8).to(DEVICE)
    
    # 4. Build W3 (8 -> 8)
    print("Building W3 descriptors...")
    desc_w3 = build_orbit_descriptors_from_cycles(cycles_Phi, cycles_Phi)
    r3, c3, pi3 = [], [], []
    for i, desc in enumerate(desc_w3):
        r, c = expand_descriptor_to_indices(desc, cycles_Phi, cycles_Phi)
        r3.append(r); c3.append(c); pi3.append(np.full(len(r), i, dtype=np.int64))
        
    rows_w3 = torch.tensor(np.concatenate(r3), dtype=torch.long, device=DEVICE)
    cols_w3 = torch.tensor(np.concatenate(c3), dtype=torch.long, device=DEVICE)
    pidx_w3 = torch.tensor(np.concatenate(pi3), dtype=torch.long, device=DEVICE)
    
    w3_params = nn.Parameter(torch.randn(len(desc_w3), device=DEVICE) * 0.01)
    bn3 = OrbitBatchNorm(cycles_Phi, 8).to(DEVICE)

    # 5. Output Projection
    L_values = [[1.0, -1.0, 1.0, -1.0, 0.0, 0.0, 0.0, 0.0],
                [0.0, 0.0, 0.0, 0.0, 1.0, -1.0, 1.0, -1.0]]
    L_matrix = torch.tensor(L_values, dtype=torch.float32, device=DEVICE)
    sigma2_layer = Sigma2(L_matrix).to(DEVICE)
    
    # 6. Model & Stats
    model = EquivariantModel(rows_w1, cols_w1, pidx_w1, bn1, drop1,
                             rows_w2, cols_w2, pidx_w2, bn2, 
                             rows_w3, cols_w3, pidx_w3, bn3,
                             sigma2_layer).to(DEVICE)

    total_params = len(w1_params) + len(w2_params) + len(w3_params) + \
                   sum(p.numel() for p in bn1.parameters()) + \
                   sum(p.numel() for p in bn2.parameters()) + \
                   sum(p.numel() for p in bn3.parameters())
    
    # 7. Data
    full_dataset = EllipseDataset(img_dir=IMAGE_DIR, preload=True)
    train_stats = full_dataset.get_label_statistics()
    
    test_dataset = None
    if os.path.exists(TEST_DIR):
        test_dataset = EllipseDataset(img_dir=TEST_DIR, preload=True)
        print(f"Loaded test dataset: {len(test_dataset)}")
    
    # 8. Save Full Config
    config = {
        'seed': SEED,
        'files': {'input_mtx': MTX_INPUT_FILENAME, 'hidden_mtx': MTX_HIDDEN_FILENAME},
        'dims': f"{N_in} -> {N_hid} -> 8 -> 8 -> 2",
        'hyperparameters': {'lr': LR, 'batch_size': BATCH_SIZE, 'epochs': N_EPOCHS},
        'regularization': {
            'dropout_rate': DROPOUT_RATE,
            'dropout_layer': 'OrbitDropout on Layer 1',
            'batch_norm': 'OrbitBatchNorm (All Layers)'
        },
        'model_params': total_params,
        'data_stats': train_stats
    }
    save_config(exp_dir, config)

    # 9. Training Loop
    all_trainable = [w1_params, w2_params, w3_params] + list(bn1.parameters()) + list(bn2.parameters()) + list(bn3.parameters())
    optimizer = optim.Adam(all_trainable, lr=LR)
    loss_function = nn.MSELoss()
    training_log = []
    best_val_loss = float('inf')
    
    if len(full_dataset) > 0:
        train_size = int(TRAIN_SPLIT * len(full_dataset))
        val_size = len(full_dataset) - train_size
        train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
        
        print(f"Starting Training ({train_size} train, {val_size} val)...")
        for epoch in range(N_EPOCHS):
            model.train()
            train_loss = 0.0
            for x_cpu, y_cpu in tqdm(train_loader, desc=f"Ep {epoch+1}", leave=False):
                x = x_cpu.to(DEVICE, non_blocking=PIN_MEMORY)
                y = y_cpu.to(DEVICE, non_blocking=PIN_MEMORY)
                optimizer.zero_grad()
                pred = model(x, w1_params, w2_params, w3_params)
                loss = loss_function(pred, y)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            
            avg_train = train_loss / len(train_loader)
            val_results = evaluate_model(model, val_loader, w1_params, w2_params, w3_params, DEVICE)
            avg_val = val_results['avg_loss']
            
            training_log.append({'epoch': epoch+1, 'train_loss': avg_train, 'val_loss': avg_val})
            print(f"Epoch {epoch+1}: Train {avg_train:.5f}, Val {avg_val:.5f}")
            
            if avg_val < best_val_loss:
                best_val_loss = avg_val
                # Save Best Checkpoint
                torch.save({
                    'epoch': epoch + 1,
                    'w1': w1_params, 'w2': w2_params, 'w3': w3_params,
                    'bn1': bn1.state_dict(), 'bn2': bn2.state_dict(), 'bn3': bn3.state_dict(),
                    'model_state': model.state_dict(),
                    'val_loss': avg_val
                }, os.path.join(exp_dir, "model_best.pt"))
        
        save_training_log(exp_dir, training_log)
        plot_learning_curves(exp_dir, training_log)
        
        # Validation Analysis
        print("Saving validation metrics...")
        val_metrics = compute_metrics(val_results)
        save_metrics(exp_dir, val_metrics, "val_metrics.json")
        save_predictions(exp_dir, val_results, "val_predictions.csv")
        plot_predictions(exp_dir, val_results, prefix="val_")
        plot_residuals(exp_dir, val_results, prefix="val_")

    # 10. Test Set Evaluation
    if test_dataset and len(test_dataset) > 0:
        print("\nEvaluating on Test Set (using best model)...")
        best_ckpt = torch.load(os.path.join(exp_dir, "model_best.pt"), map_location=DEVICE)
        
        # Restore Best Params
        w1_params.data = best_ckpt['w1'].to(DEVICE)
        w2_params.data = best_ckpt['w2'].to(DEVICE)
        w3_params.data = best_ckpt['w3'].to(DEVICE)
        bn1.load_state_dict(best_ckpt['bn1'])
        bn2.load_state_dict(best_ckpt['bn2'])
        bn3.load_state_dict(best_ckpt['bn3'])
        
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
        test_results = evaluate_model(model, test_loader, w1_params, w2_params, w3_params, DEVICE)
        test_metrics = compute_metrics(test_results)
        
        save_metrics(exp_dir, test_metrics, "test_metrics.json")
        save_predictions(exp_dir, test_results, "test_predictions.csv")
        plot_predictions(exp_dir, test_results, prefix="test_")
        plot_residuals(exp_dir, test_results, prefix="test_")
        print(f"Test MSE: {test_metrics['combined_mse']:.6f}")

    # 11. Equivariance Check
    print("\nVerifying Equivariance...")
    sigma_inv = [0]*N_in
    for i, s in enumerate(sigma_in): sigma_inv[s] = i
    sigma_idx = torch.tensor(sigma_inv, device=DEVICE, dtype=torch.long)
    R_target = torch.tensor([[-1., 0.], [0., -1.]], device=DEVICE)
    
    model.eval()
    with torch.no_grad():
        x = torch.randn(1, N_in, device=DEVICE)
        out_x = model(x, w1_params, w2_params, w3_params).squeeze(0)
        x_G = torch.index_select(x, 1, sigma_idx)
        out_Gx = model(x_G, w1_params, w2_params, w3_params).squeeze(0)
        rotated_out = R_target @ out_x
        diff = (out_Gx - rotated_out).norm().item()
        
        equiv_check = {
            'passed': diff < 1e-4, 'diff': diff,
            'out_x': out_x.tolist(), 'out_Gx': out_Gx.tolist()
        }
        with open(os.path.join(exp_dir, "equivariance_check.json"), 'w') as f:
            json.dump(equiv_check, f, indent=2)
            
        if diff < 1e-4: print("SUCCESS: Equivariance maintained.")
        else: print(f"WARNING: Equivariance broken (diff={diff:.4e}).")

    # 12. Final Checkpoint & Notes
    torch.save({
        'w1': w1_params, 'w2': w2_params, 'w3': w3_params,
        'state': model.state_dict()
    }, os.path.join(exp_dir, "model_final.pt"))

    notes = f"""# Experiment Notes
- Date: {datetime.datetime.now()}
- Model: {MODEL_NAME}
- Best Val Loss: {best_val_loss:.6f}
- Equivariance Diff: {diff:.6e}
- Input: 250x250
"""
    save_notes(exp_dir, notes)
    print(f"\nExperiment Complete. Saved to: {exp_dir}")

if __name__ == '__main__':
    main()